In [3]:
import os
import glob
import cv2
import numpy as np
from ultralytics import YOLO

# ==========================================
# 🎛️ 控制台 (Control Panel)
# ==========================================

# 1. 來源選擇 (Source Selection)
SELECT_INDEX = 4  # <--- 目前設定為 YouTube 清單中的第四個 (可改為 1, 2, 3, 4 或檔案路徑)

# 預設的 YouTube 清單
YOUTUBE_LIST = {
    1: "https://www.youtube.com/watch?v=RzNwV-gHheM",
    2: "https://www.youtube.com/watch?v=tCYsrTm6zQU",
    3: "https://www.youtube.com/watch?v=x_gEptGW02U",
    4: "https://www.youtube.com/watch?v=paNBwxZsoj0"
}

# 自動設定 INPUT_SOURCE
if SELECT_INDEX in YOUTUBE_LIST:
    INPUT_SOURCE = YOUTUBE_LIST[SELECT_INDEX]
else:
    INPUT_SOURCE = SELECT_INDEX

# ------------------------------------------

# 2. 核心功能
ENABLE_SAVING = True
ENABLE_SHOWING = True 

# 3. 速度控制
VID_STRIDE = 3          # 步幅：1=每幀都測

# 4. 偵測設定
CONF_THRESHOLD = 0.5    # 信心度門檻
IMG_SIZE = 800          # 推論解析度
NMS_IOU = 0.8           # 非最大抑制 (NMS) IoU 門檻

# 5. 視覺設定
LINE_WIDTH = 3
FONT_SIZE = 1.2
SHOW_LABELS = True
SHOW_CONF = True

# ==========================================
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
# ==========================================

def find_latest_model_path(base_dir='runs/detect'):
    search_path = os.path.join(base_dir, 'train*')
    dirs = glob.glob(search_path)
    if not dirs: raise FileNotFoundError(f"❌ 找不到訓練資料夾於 {base_dir}")
    latest_dir = max(dirs, key=os.path.getmtime)
    weights_path = os.path.join(latest_dir, 'weights', 'best.pt')
    if not os.path.exists(weights_path): raise FileNotFoundError(f"❌ 找不到 {weights_path}")
    print(f"✅ 自動載入權重: {weights_path}")
    return weights_path

def determine_input_type(source):
    if isinstance(source, int): return 'webcam'
    if isinstance(source, str):
        if source.startswith("http") or "youtube.com" in source or "youtu.be" in source:
            return 'youtube'
        if not os.path.exists(source): raise FileNotFoundError(f"❌ 找不到檔案: {source}")
        ext = os.path.splitext(source)[1].lower()
        if ext in IMAGE_EXTENSIONS: return 'image'
        return 'video'
    return 'unknown'

def process_image(model, source_path):
    print(f"🖼️ 正在處理圖片 (靜默模式): {source_path}...")
    frame = cv2.imread(source_path)
    if frame is None: return

    results = model(frame, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False)
    
    annotated_frame = results[0].plot(
        line_width=LINE_WIDTH, 
        font_size=FONT_SIZE, 
        labels=SHOW_LABELS, 
        conf=SHOW_CONF
    )
    
    if ENABLE_SAVING:
        base_name, ext = os.path.splitext(source_path)
        output_path = f"{base_name}_detected{ext}"
        cv2.imwrite(output_path, annotated_frame)
        print(f"💾 圖片處理完畢，已儲存至: {output_path}")
    
    print("ℹ️ (已略過視窗顯示)")

def process_video_or_stream(model, source, input_type):
    print(f"📹 正在處理 {input_type} (步幅={VID_STRIDE}): {source}...")

    cap = None
    if input_type == 'youtube':
        try:
            from cap_from_youtube import cap_from_youtube
            cap = cap_from_youtube(source, resolution='720p') 
        except Exception as e:
            print(f"❌ YouTube 載入失敗: {e}"); return
    else:
        cap = cv2.VideoCapture(source)

    if not cap.isOpened(): print(f"❌ 無法開啟來源"); return

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30
    
    save_fps = fps / VID_STRIDE
    print(f"ℹ️ 原始 FPS: {fps:.2f}, 輸出 FPS: {save_fps:.2f}")

    out = None
    if ENABLE_SAVING:
        if input_type == 'youtube': output_path = "youtube_detected.mp4"
        elif input_type == 'webcam': output_path = "webcam_detected.mp4"
        else:
            base_name = os.path.splitext(os.path.basename(source))[0]
            output_path = f"{base_name}_detected.mp4"
            
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
        out = cv2.VideoWriter(output_path, fourcc, save_fps, (w, h))
        print(f"💾 影片將儲存至: {output_path}")

    print("\n🚀 開始偵測... (按 'q' 離開)")

    frame_count = 0
    while True:
        success, frame = cap.read()
        if not success: break
        
        frame_count += 1
        if frame_count % VID_STRIDE != 0:
            continue

        results = model.track(
            frame, 
            imgsz=IMG_SIZE, 
            conf=CONF_THRESHOLD,
            tracker='botsort.yaml',
            iou=NMS_IOU,
            persist=True,           
            verbose=False
        )
        
        annotated_frame = results[0].plot(
            line_width=LINE_WIDTH, font_size=FONT_SIZE,
            labels=SHOW_LABELS, conf=SHOW_CONF
        )

        if ENABLE_SHOWING:
            cv2.imshow(f"YOLOv10 {input_type}", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
        elif not ENABLE_SAVING:
            print(".", end="", flush=True)
        
        if ENABLE_SAVING and out is not None:
            out.write(annotated_frame)

    cap.release()
    if out is not None: out.release()
    cv2.destroyAllWindows()

def main():
    try:
        if 'NMS_IOU' not in globals():
            raise NameError("NMS_IOU 變數未在 Control Panel 中定義。")
            
        model_path = find_latest_model_path()
        model = YOLO(model_path)
        input_type = determine_input_type(INPUT_SOURCE)
        
        if input_type == 'image':
            process_image(model, INPUT_SOURCE)
        else:
            process_video_or_stream(model, INPUT_SOURCE, input_type)
            
        print("\n✅ 處理完成！")
    except Exception as e:
        print(f"\n❌ 發生錯誤: {e}")

if __name__ == "__main__":
    main()

In [1]:
# ==========================================
# --- 批量自動偵測版：多影片處理 + 統計報告 ---
# ==========================================
import os, sys, glob, cv2, time
import numpy as np
from ultralytics import YOLO
from tqdm.notebook import tqdm
from collections import defaultdict

# --- 1. 批量任務清單 ---
TARGET_VIDEOS = [
    r"c:\Users\0419mch\Desktop\project_0414\測試模型用\模擬_1.mp4",
    r"c:\Users\0419mch\Desktop\project_0414\測試模型用\模擬_2.mp4",
    r"c:\Users\0419mch\Desktop\project_0414\測試模型用\模擬_3.mp4"
]

# --- 2. 偵測設定 ---
CUSTOM_TRAIN_ID = 7
IMG_SIZE       = 640
CONF_THRESHOLD = 0.5
VID_STRIDE     = 1
ENABLE_SAVING  = True
ENABLE_SHOWING = False

def _run_batch_inference(model, video_list):
    print(f"準備開始批次任務，共 {len(video_list)} 支影片")
    
    main_pbar = tqdm(video_list, desc="Batch Progress")
    
    for video_path in main_pbar:
        if not os.path.exists(video_path):
            print(f"Skip: {video_path}")
            continue
            
        video_name = os.path.basename(video_path)
        main_pbar.set_description(f"Processing: {video_name}")
        
        cap = cv2.VideoCapture(video_path)
        out = None
        class_stats = defaultdict(list)
        frame_count = 0
        
        try:
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            if ENABLE_SAVING:
                save_path = video_path.replace('.mp4', '_detected.mp4')
                out = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/VID_STRIDE, (w, h))
            
            sub_pbar = tqdm(total=total_frames, desc=f"  > Frames", leave=False)
            
            while True:
                success, frame = cap.read()
                if not success: break
                
                frame_count += 1
                sub_pbar.update(1)
                if frame_count % VID_STRIDE != 0: continue

                results = model.track(frame, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, persist=True, verbose=False)
                annotated = results[0].plot()

                current_classes = results[0].boxes.cls.cpu().numpy().astype(int)
                for cls_id in model.names.keys():
                    count = np.sum(current_classes == cls_id)
                    class_stats[cls_id].append(count)

                total_now = len(results[0].boxes)
                overlay = annotated.copy()
                cv2.rectangle(overlay, (10, 10), (420, 80), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0, annotated)
                cv2.putText(annotated, f"Total: {total_now}", (25, 55), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

                if out: out.write(annotated)
                
            sub_pbar.close()
            
            print(f"Summary for {video_name}:")
            for cls_id, name in model.names.items():
                counts = class_stats[cls_id]
                if counts:
                    print(f"  [{name}]: Max={max(counts)}, Avg={sum(counts)/len(counts):.2f}")
            
        finally:
            cap.release()
            if out: out.release()
            cv2.destroyAllWindows()

best_pt = os.path.join("runs", "detect", f"train{CUSTOM_TRAIN_ID}", "weights", "best.pt")
if os.path.exists(best_pt):
    model = YOLO(best_pt)
    _run_batch_inference(model, TARGET_VIDEOS)
    print("Batch complete!")
else:
    print("Weights not found")


準備開始批次任務，共 3 支影片


Batch Progress:   0%|          | 0/3 [00:00<?, ?it/s]

  > Frames:   0%|          | 0/192 [00:00<?, ?it/s]

Summary for 模擬_1.mp4:
  [level_1]: Max=7, Avg=2.80
  [level_2]: Max=13, Avg=6.82
  [level_3]: Max=3, Avg=0.98


  > Frames:   0%|          | 0/192 [00:00<?, ?it/s]

Summary for 模擬_2.mp4:
  [level_1]: Max=4, Avg=1.12
  [level_2]: Max=7, Avg=1.33
  [level_3]: Max=4, Avg=1.02


  > Frames:   0%|          | 0/192 [00:00<?, ?it/s]

Summary for 模擬_3.mp4:
  [level_1]: Max=4, Avg=1.10
  [level_2]: Max=4, Avg=0.55
  [level_3]: Max=2, Avg=0.51
Batch complete!
